# Recorded FiLM, action and EE oscillations
Do larger or more periodic modulated-feature variations co-occur with action variation and EE oscillations during sustained loading? This workflow uses recorded signals; checkpoints are needed only for the optional unmodulated-action comparison.


In [ ]:
from pathlib import Path
import os, json, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'notebooks').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'notebooks'))
import exp3_film_oscillation as analysis
CONFIG = {
    'recording_roots': [str(ROOT / 'evaluation/exp2')],
    'tasks': ['b1z1_pact'], 'checkpoints': None,
    'profiles': ['pushing', 'pulling'], 'loading': ['loaded', 'unloaded'],
    'evaluation_seeds': [0, 1, 2], 'trial_ids': None,
    'output_dir': str(ROOT / 'analysis/exp3_film_recorded'),
    'sustained_window_seconds': 4., 'minimum_window_seconds': 1.25,
    'transient_margin_seconds': .25, 'frequency_band_hz': [.5, 12.],
    'nperseg': 64, 'max_lag_seconds': .5,
    'feature_components': 3, 'feature_basis_samples': 2048, 'sampling_seed': 0,
    'num_workers': 10,  # Try 2-4 CPU window workers; at most 2*workers windows in flight.
    'cpu_blas_threads': 1,  # Also bound the parent PCA/SVD thread team (threadpoolctl).
    'cache_enabled': True,  # Reuse basis/window calculations when only plots change.
    'cache_dir': str(ROOT / 'analysis/exp3_numeric_cache'),
    'unmodulated_comparison': True, 
    'checkpoint_paths': {},
    'physical_comparison': True,
    'comparison_tasks': ['b1z1_pact','b1z1_unifp_original','b1z1_ppo_pos'],
    'max_plot_windows': 6, 'max_scatter_plots': 6, 'max_windows': None,
    'pca': True,  # Optional supporting embedding visualization; temporal projections always share a fixed basis.
    'figure_title_font_size': 12, 'title_font_size': 11,
    # Publication figures use compact numeric outputs; no checkpoint access.
    'oscillation_axes': ['x', 'y', 'z'],  # Both figures and tables are exported under axis-x/y/z.
    'oscillation_selection': {'profile': 'pushing', 'loading': 'loaded', 'motion_phase': 'moving'},  # Add stage/checkpoint/task to select a cohort.
    'oscillation_example': {},  # Optional evaluation_seed/trial_id/window_id; otherwise median trial amplitude.
    'oscillation_pcs': [1, 2], 'oscillation_ee_axis': 'dominant',  # Single-axis API only; the notebook uses oscillation_axes.
    'oscillation_arm_joints': None,  # First two recorded arm names; explicit names recommended.
    'oscillation_leg_joints': [],  # Optional A/B traces; C retains selected arm channels.
    'paired_pca_selection': {'loading': 'loaded'},
    'paired_pca_error_axis': 'x', 'paired_pca_error_limit_m': None,  # Axis is overridden for each notebook figure set.
    'paired_pca_points_per_facet': 30, 'paired_pca_max_facets': 6,
    'axis_label_font_size': 10, 'tick_font_size': 9, 'legend_font_size': 8, 'annotation_font_size': 8,
}
CONFIG.update(json.loads(os.environ.get('FILM_OSCILLATION_CONFIG', '{}')))


## Signals and averaging
EE errors are signed measured-minus-reference Cartesian errors in the yaw-aligned task frame [m]. Features/actions are actor-time signals; successor signals, where used, retain the +dt offset. Position commands are scaled offsets [rad], and FF commands [Nm] precede motor/branch weighting. Executed torques remain separate.

Windows never cross resets, failures or stage boundaries. Bias is the window mean; core oscillatory amplitudes and spectra remove that mean consistently. A bounded seeded pooled pre/post PCA basis is fitted once per checkpoint and reused for signed temporal projections. No neuron alignment across checkpoints is implied. Coherence requires enough Welch averages; phase is shown only above the coherence threshold. Short-window frequency limits are exported.

Window metrics average within trials before evaluation-seed summaries. Across-trial amplitude correlations stay within checkpoint/task/load/motion stage and evaluation seed; constant signals or fewer than three trials give unavailable correlations. Evaluation seeds are not independent training checkpoints.

**Runtime:** start with `num_workers=1`, then try 2-4 if CPU calculations dominate. Spawned CPU workers use one BLAS thread and bounded, ordered window jobs; aggregation/plotting stay in the parent. More workers require more RAM and may not help slow storage. `timings.json` separates discovery, basis, numerical/export, aggregation, example plots and the unchanged physical comparison. Caches contain numeric arrays/JSON, not models; they invalidate on numerical inputs, metadata and relevant code changes. Basis discovery also checks immutable chunk size/mtime. Use a fresh `output_dir` with the same `cache_dir` to change plots without repeating numerical calculations (recording reads/aggregation still run). Caches can grow large; delete `cache_dir` to reclaim space or invalidate it after external dynamics-library upgrades.


In [ ]:
result = analysis.run_analysis(CONFIG)
display(pd.Series(result['timings'], name='Runtime / cache counts'))
OUT = Path(CONFIG['output_dir'])
summary = pd.DataFrame(result['summary'])
if len(summary):
    selected = ['pre/oscillatory_rms','post/oscillatory_rms','post_pre_oscillation_ratio','modulation/rms','modulation/oscillatory_rms',
                'position_recorded_arm/oscillatory_rms','position_recorded_leg/oscillatory_rms',
                'torque_recorded_arm/oscillatory_rms','torque_recorded_leg/oscillatory_rms']
    display(summary[summary.metric.isin(selected) | summary.metric.str.contains('ee_cartesian_error/component_.*oscillatory_rms')])
    temporal = summary[summary.metric.str.startswith('association/') & summary.metric.str.contains('zero_lag_correlation|peak_correlation|coherence_at_ee_')]
    display(temporal)
display(pd.DataFrame(result['amplitude_associations']))
missing = pd.DataFrame(result['exclusions'])
if len(missing): display(missing.groupby('reason').size().rename('affected_windows_or_runs').reset_index())
print('Exports:', OUT)


In [ ]:
for selected in result['plot_selection']:
    for suffix in ('traces','spectra'):
        display(Image(filename=str(OUT / f"window-{selected['window_id']:05d}-{suffix}.png")))
for path in sorted(OUT.glob('amplitudes-*.png')): display(Image(filename=str(path)))
physical = pd.DataFrame(result.get('physical_comparison', {}).get('matched', []))
if len(physical): display(physical[(physical.signal == 'ee_error') & physical.metric.isin(['oscillatory_rms','peak_to_peak','band_power','dominant_frequency'])])


## Feature-action-EE oscillations and paired PCA
Both figures are generated for x, y and z under `axis-x`, `axis-y` and `axis-z`. Each axis uses its own median-amplitude example unless explicitly overridden; the PCA basis and paired samples remain identical across axes.
A uses a contiguous sustained window near the median trial amplitude. B uses the same saved component spectra and frequency bins, preserving physical action PSD amplitudes. C uses all eligible trial means in that task/load/motion stage, separately for position and FF channels; markers identify evaluation seeds. Spearman correlations describe associations across trial means, not independent training replicates. PCs share one checkpoint-wide pooled basis; signed yaw-frame EE measurements are aligned at actor time t (not successor t+dt). Missing unmodulated actions are omitted.

Paired PCA facets share that basis, axis limits and signed-error color scale. Hollow/filled markers are pre/post-FiLM features; arrows connect the same timestep, not a temporal trajectory. Subsampling is deterministic and bounded per facet. Change selectors above and rerun this cell alone to reuse window exports, without models or raw-feature rereads. Trial statistics and selection metadata accompany PDF/PNG figures.


In [ ]:
import exp3_film_figures as film_figures
OUT = Path(CONFIG['output_dir'])
if not (OUT / 'COMPLETE').exists(): raise ValueError('Select a completed FiLM analysis output directory')
publication_source = json.loads((OUT / 'summary.json').read_text())  # This cell also works without rerunning analysis.
publications = film_figures.export_all_axes(publication_source['windows'], {**publication_source['config'], **CONFIG}, OUT)
for axis, publication in publications.items():
    print(f'EE {axis.upper()} axis')
    axis_output = OUT / ('axis-' + axis)
    for stem in publication['figures']: display(Image(filename=str(axis_output / (stem + '.png'))))
    display(pd.DataFrame(publication['diagnostics']))
    if publication['selection'] is not None: display(pd.read_csv(axis_output / 'oscillation_plotted_trials.csv'))


## Optional unmodulated actions
Set `unmodulated_comparison=True` and supply checkpoint overrides only if needed. The same frozen position/torque heads evaluate recorded pre/post features, preserving action order and scaling. Reconstruction errors are reported in physical command units. Unmodulated actions are evaluated on recorded features; they are not a separately executed robot trajectory.


In [ ]:
if CONFIG.get('unmodulated_comparison'):
    if len(summary):
        display(summary[summary.metric.str.contains('position_(actual|unmodulated|film)|torque_(actual|unmodulated|film)', regex=True)])
        change = summary[summary.metric.str.endswith('/modulated_minus_unmodulated_rms')].copy()
        change['fluctuation_change'] = change['mean'].map(lambda x: 'increases' if x > 1e-8 else 'decreases' if x < -1e-8 else 'preserves')
        display(change)
    errors = [dict(window=w['window_id'], **{k:v for k,v in w['units'].items() if k.endswith('reconstruction_max_error')})
              for w in result['windows']]
    display(pd.DataFrame(errors))


## Outputs
`threadpoolctl` is optional. Without it, analysis and workers continue with a warning, but runtime BLAS thread limits cannot be guaranteed. Install it in this kernel with `%pip install threadpoolctl` for thread-count control.
CSV/JSON retain amplitudes, signed component metrics, temporal relationships, trial/seed counts and missing data. `trial_amplitudes.csv` and `amplitude_associations.csv` retain within-stage trial comparisons; numeric window NPZs and PDF/PNG figures support inspection. Physical comparisons use matching recorded protocols and identical spectral bands. Use a fresh output directory. CPU dependencies: NumPy, SciPy, pandas, matplotlib, Jupyter; optional PyTorch only for head weights. No simulator is initialized.
